In [14]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
# Path
PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

In [15]:
from utils.data_loadingV4_file import load_data as load_v4_file, load_test_data
from models.vanilla_rnn import VanillaRNN
from sklearn.metrics import mean_absolute_error
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

In [16]:
CONFIG = {
    'TRAIN_PATH': PROJECT_ROOT / "datasets_challenge" / "train",
    'VAL_RATIO': 0.2,
    'BATCH_SIZE': 64,
    'COLUMNS_TO_DROP': ['sideSlip', 'vyCG', 'longitude', 'latitude', 'time'],
    'SEQUENCE_LENGTH': 20,
    'THRESHOLD': True,
    'THRESHOLD_VALUE': 3.0,
    'THRESHOLD_COLUMN': 'vxCG',
    'TARGET': 'sideSlip',
    'SCALING': True,
    'SEED': 42,
    'DIST_CSV': PROJECT_ROOT / "datasets_challenge" / "sideSlip_distribution.csv",
}

In [17]:
print("\nLoading V4: File-Based Split (File 7)")
train_v4, val_v4, scaler_v4, _, _ = load_v4_file(
    train_path=CONFIG['TRAIN_PATH'],
    val_file_num=17,
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    batchsize=CONFIG['BATCH_SIZE'],
    seed=CONFIG['SEED']
)

def extract_targets(loader):
    targets = []
    for _, y in loader:
        targets.append(y)
    return torch.cat(targets).numpy().flatten()

train_targets_v4 = extract_targets(train_v4)
val_targets_v4 = extract_targets(val_v4)
print(f"V4 File:     Train {len(train_targets_v4):,} | Val {len(val_targets_v4):,}")


Loading V4: File-Based Split (File 7)
V3 Data Loader: File-Based Split

Found 22 files
Using train_part_017.csv for validation

  Val:   train_part_017.csv - 2 groups

Total: 40 train groups, 2 val groups

Band Distribution:
Train: Many=91.2%, Medium=6.9%, Few=1.9%
Val:   Many=55.6%, Medium=23.0%, Few=21.4%

Train: 136,559 sequences | Val: 521 sequences

V4 File:     Train 136,559 | Val 521


In [18]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = VanillaRNN(
    input_size=22,          # number of features
    hidden_size=48,
    num_layers=1,
    dropout=0.0
).to(device)

criterion = nn.L1Loss()     # MAE (matches few-shot objective)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

In [19]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for X, y in dataloader:
        X = X.to(device)
        y = y.to(device).unsqueeze(1)

        optimizer.zero_grad()
        preds = model(X)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [20]:
def validate(model, dataloader, device):
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            pred = model(X).squeeze(1)
            preds.append(pred.cpu().numpy())
            targets.append(y.cpu().numpy())

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

    return mean_absolute_error(targets, preds)


In [21]:
EPOCHS = 30
best_val_mae = float("inf")
best_model_state = None

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        model,
        train_v4,  
        optimizer,
        criterion,
        device 
    )

    val_mae = validate(
        model,
        val_v4,     # ← no sampler, true distribution
        device
    )

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val MAE: {val_mae:.4f}"
    )

    # Save best model (important for snapshot / ensemble)
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_model_state = model.state_dict()
model.load_state_dict(best_model_state)
print(f"✓ Best validation MAE: {best_val_mae:.4f}")   

Epoch 01 | Train Loss: 0.2882 | Val MAE: 0.4221
Epoch 02 | Train Loss: 0.1920 | Val MAE: 0.3141
Epoch 03 | Train Loss: 0.1760 | Val MAE: 0.3144
Epoch 04 | Train Loss: 0.1679 | Val MAE: 0.3466
Epoch 05 | Train Loss: 0.1636 | Val MAE: 0.2786
Epoch 06 | Train Loss: 0.1590 | Val MAE: 0.3151
Epoch 07 | Train Loss: 0.1562 | Val MAE: 0.2917
Epoch 08 | Train Loss: 0.1538 | Val MAE: 0.3383
Epoch 09 | Train Loss: 0.1519 | Val MAE: 0.4038
Epoch 10 | Train Loss: 0.1496 | Val MAE: 0.2641
Epoch 11 | Train Loss: 0.1482 | Val MAE: 0.3057
Epoch 12 | Train Loss: 0.1471 | Val MAE: 0.2975
Epoch 13 | Train Loss: 0.1455 | Val MAE: 0.3402
Epoch 14 | Train Loss: 0.1449 | Val MAE: 0.3250
Epoch 15 | Train Loss: 0.1438 | Val MAE: 0.3074
Epoch 16 | Train Loss: 0.1429 | Val MAE: 0.2968
Epoch 17 | Train Loss: 0.1417 | Val MAE: 0.2206
Epoch 18 | Train Loss: 0.1411 | Val MAE: 0.2547
Epoch 19 | Train Loss: 0.1404 | Val MAE: 0.2459
Epoch 20 | Train Loss: 0.1396 | Val MAE: 0.2853
Epoch 21 | Train Loss: 0.1386 | Val MAE:

In [27]:
print("\nLoading V4: File-Based Split (File 7)")
train_v4, val_v4, scaler_v4, _, _ = load_v4_file(
    train_path=CONFIG['TRAIN_PATH'],
    val_file_num=17,
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    batchsize=CONFIG['BATCH_SIZE'],
    seed=CONFIG['SEED'],
    shuffle= False
)


Loading V4: File-Based Split (File 7)
V3 Data Loader: File-Based Split

Found 22 files
Using train_part_017.csv for validation

  Val:   train_part_017.csv - 2 groups

Total: 40 train groups, 2 val groups

Band Distribution:
Train: Many=91.2%, Medium=6.9%, Few=1.9%
Val:   Many=55.6%, Medium=23.0%, Few=21.4%

Train: 136,559 sequences | Val: 521 sequences



In [28]:
import plotly.graph_objects as go

model.eval()

preds = []
ground_truths = []
total_loss = 0

with torch.no_grad():
    for step, (x, y) in enumerate(val_v4):
        X = x.to(device)
        y = y.to(device).reshape(-1, 1)

        pred = model(X)
        loss = criterion(pred, y)
        total_loss += loss.item()

        preds.append(pred.reshape(-1).cpu().numpy())
        ground_truths.append(y.reshape(-1).cpu().numpy())

print(f"Avg Loss: {total_loss / (step + 1)}")

preds = np.concatenate(preds)
ground_truths = np.concatenate(ground_truths)

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=ground_truths,
    mode='lines',
    name='truth',
    line=dict(color='blue')
))
fig.add_trace(go.Scatter(
    y=preds,
    mode='lines',
    name='pred',
    line=dict(color='red'),
    opacity=0.8
))

fig.update_layout(
    title='Ground Truth vs Predictions (TRAIN)',
    xaxis_title='Sample Index',
    yaxis_title='Sideslip Angle (degrees)',
    width=1200,
    height=500
)

fig.show()


Avg Loss: 0.2652778596513801


In [29]:
print("\nLoading test data...")
test_dataloader = load_test_data(test_path = PROJECT_ROOT / "datasets_challenge" / "test",
                                 batchsize = 512,
                                 scaler= scaler_v4,
                                 columns_to_drop= CONFIG["COLUMNS_TO_DROP"])


Loading test data...
📂 Loading Test Data

Found 11 test files

✅ Test sequences: 49,328
   ✅ Matches Kaggle requirement!



In [30]:
model.eval()
test_preds = []

with torch.no_grad():
    for step, X in enumerate(test_dataloader):
        X = X.to(device)

        pred = model(X)

        test_preds.append(pred.reshape(-1).cpu().numpy())

test_preds = np.concatenate(test_preds)
print("Shape of Predictions:", test_preds.shape)


Shape of Predictions: (49328,)


In [31]:
# Save predictions in the required format for submission
y_pred = pd.DataFrame({"predicted_sideslip": test_preds})
y_pred["id"] = range(len(test_preds))  # Add sample IDs (0 to 49327)

# Save to CSV file
y_pred.to_csv("predictionsfilebased.csv", index=False)

print(f"✓ Predictions saved to 'predictions.csv'")
print(f"✓ Total samples: {len(test_preds)}")
print(f"✓ Ready for submission to the competition!")

✓ Predictions saved to 'predictions.csv'
✓ Total samples: 49328
✓ Ready for submission to the competition!


### Got a kaggle score of 0.2859